In [1]:
cd ..

/home/serafeim/Desktop/bet


In [2]:

from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.last_activity_generator import generate_last_activity
from prepare_data_inference import prepare_data_inference


In [3]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()

player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/02 21:22:38 WARN Utils: Your hostname, serafeim-virtual-machine, resolves to a loopback address: 127.0.1.1; using 10.100.2.34 instead (on interface ens192)
26/03/02 21:22:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 21:22:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/02 21:22:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/02 21:22:50 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')


In [5]:
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')

In [6]:
data_inference = prepare_data_inference('2024-06-20')

failed_withdrawals_30d
deposit_count_30d
withdrawal_count_30d
withdrawal_ratio


In [7]:
player_behavior.filter(F.col('reference_date')=='2024-06-20').count()

655

In [10]:
data_inference.count()


652

In [11]:
ex_ids = player_behavior.filter(F.col('reference_date')=='2024-06-20').select('player_idx').join(data_inference.select('player_idx'), on='player_idx', how='left_anti')
ex_ids.show()

+----------+
|player_idx|
+----------+
|       765|
|       748|
|       889|
|       848|
+----------+



In [12]:
ex_frame = player_behavior.filter(F.col('reference_date')=='2024-06-20').join(ex_ids, on='player_idx', how='inner')
ex_frame.show()

+----------+--------------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|player_idx|reference_date|    balance_7d_ago|   balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|
+----------+--------------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|       765|    2024-06-20|            1076.7|            1076.7|                 0.0

In [13]:
data_inference.select('player_idx').join(player_behavior.filter(F.col('reference_date')=='2024-06-20').select('player_idx'), on='player_idx', how='left_anti').show()

+----------+
|player_idx|
+----------+
|       713|
+----------+



In [ ]:
silver_money_events.filter(F.col('player_idx')==713).orderBy(F.col('event_ts').desc()).show(truncate=False)

In [ ]:
ex_frame.filter(F.col('deposit_count_30d')==1).show()

In [15]:
from pyspark.sql.types import NumericType

In [16]:
# Identify numeric columns
numeric_cols = [
    field.name
    for field in ex_frame.schema.fields
    if isinstance(field.dataType, NumericType)
]

# Compute sums
df_sum = ex_frame.select([
    F.sum(F.col(c)).alias(c)
    for c in numeric_cols
])

df_sum.show()

+----------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|player_idx|    balance_7d_ago|   balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|
+----------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|      3250|2448.3900000000003|2448.3900000000003|                 0.0|                  0.0|              0|               0.0|  